In [ ]:
#3D AIRFLOW STIMULATION

import math
import random
import sys
import numpy as np
import pygame
import trimesh

# ==========================================
# PART 1: 3D CAD MODEL GENERATION (TRIMESH)
# ==========================================

COLOR_WOOD = [185, 125, 68, 255]
COLOR_PLASTIC_DARK = [35, 35, 35, 255]
COLOR_HEADER_METAL = [45, 48, 52, 255]
COLOR_STAINLESS_STEEL = [195, 200, 205, 255]
COLOR_INTAKE_MESH = [160, 165, 170, 210]
COLOR_ZEOLITE = [235, 235, 225, 255]
COLOR_BIOCHAR = [15, 15, 15, 255]
COLOR_VOC_PAD = [200, 150, 95, 255]
COLOR_ONION = [225, 140, 45, 255]
COLOR_ONION_STEM = [170, 100, 25, 255]


def box(ext, pos, color):
    m = trimesh.creation.box(extents=ext)
    m.apply_translation(pos)
    m.visual.face_colors = color
    return m


def cyl(r, h, pos, color):
    m = trimesh.creation.cylinder(radius=r, height=h)
    m.apply_translation(pos)
    m.visual.face_colors = color
    return m


def cone(r, h, pos, color):
    m = trimesh.creation.cone(radius=r, height=h)
    m.apply_translation(pos)
    m.visual.face_colors = color
    return m


def perforated_plate(
    extents, pos, hole_r=0.012, grid_size=(8, 8), color=COLOR_STAINLESS_STEEL
):
    # Fast non-blocking plate creation without heavy boolean subtraction dependencies
    plate = trimesh.creation.box(extents=extents)
    plate.apply_translation(pos)
    plate.visual.face_colors = color
    return plate


def onion(r, pos):
    b = trimesh.creation.icosphere(radius=r, subdivisions=2)
    b.apply_transform(np.diag([1.0, 1.0, 0.75, 1.0]))
    b.visual.face_colors = COLOR_ONION

    t = trimesh.creation.cone(radius=r * 0.25, height=r * 0.5)
    t.apply_translation([0, 0, r * 0.65])
    t.visual.face_colors = COLOR_ONION_STEM

    o = trimesh.util.concatenate([b, t])
    o.apply_translation(pos)
    return o


def generate_3d_geometry():
    print("Building 3D Mesh Geometry...")
    all_meshes = []

    # 1. Base Wooden Crate
    all_meshes.append(box([1.22, 0.08, 0.04], [0, -0.54, 0.02], COLOR_WOOD))
    all_meshes.append(box([1.22, 0.08, 0.04], [0, 0.54, 0.02], COLOR_WOOD))
    all_meshes.append(box([0.08, 1.00, 0.04], [-0.54, 0, 0.02], COLOR_WOOD))
    all_meshes.append(box([0.08, 1.00, 0.04], [0.54, 0, 0.02], COLOR_WOOD))

    # 2. Sorbent Inlet
    inlet_z_start, inlet_height = 0.04, 0.08
    z_inlet_mid = inlet_z_start + (inlet_height / 2.0)
    wall_thick, full_w = 0.03, 1.20

    all_meshes.append(
        box(
            [full_w, full_w, 0.015],
            [0, 0, inlet_z_start + 0.0075],
            COLOR_PLASTIC_DARK,
        )
    )

    win_w, win_h, mesh_thick = 0.38, 0.045, 0.005
    for x_off in [-0.28, 0.28]:
        all_meshes.append(
            box(
                [win_w, mesh_thick, win_h],
                [x_off, -full_w / 2.0, z_inlet_mid],
                COLOR_INTAKE_MESH,
            )
        )
        all_meshes.append(
            box(
                [win_w, mesh_thick, win_h],
                [x_off, full_w / 2.0, z_inlet_mid],
                COLOR_INTAKE_MESH,
            )
        )

    for y_off in [-0.28, 0.28]:
        all_meshes.append(
            box(
                [mesh_thick, win_w, win_h],
                [-full_w / 2.0, y_off, z_inlet_mid],
                COLOR_INTAKE_MESH,
            )
        )
        all_meshes.append(
            box(
                [mesh_thick, win_w, win_h],
                [full_w / 2.0, y_off, z_inlet_mid],
                COLOR_INTAKE_MESH,
            )
        )

    for x in [-0.585, 0.0, 0.585]:
        for y in [-0.585, 0.585]:
            all_meshes.append(
                box(
                    [0.10, wall_thick, win_h],
                    [x, y, z_inlet_mid],
                    COLOR_PLASTIC_DARK,
                )
            )

    for y in [-0.585, 0.0, 0.585]:
        for x in [-0.585, 0.585]:
            all_meshes.append(
                box(
                    [wall_thick, 0.10, win_h],
                    [x, y, z_inlet_mid],
                    COLOR_PLASTIC_DARK,
                )
            )

    # 3. Sub-Mesh Layer
    z_sub_mesh = inlet_z_start + inlet_height
    Y_C, Y_L = -0.02, 1.12
    all_meshes.append(
        perforated_plate([1.14, Y_L + 0.02, 0.006], [0, Y_C - 0.01, z_sub_mesh])
    )

    # 4. Frame & Slats
    for x, y in [
        (-0.575, -0.575),
        (-0.575, 0.575),
        (0.575, -0.575),
        (0.575, 0.575),
    ]:
        all_meshes.append(
            box([0.05, 0.05, 0.96], [x, y, z_sub_mesh + 0.48], COLOR_WOOD)
        )

    for z_slat in np.linspace(z_sub_mesh + 0.05, z_sub_mesh + 0.90, 10):
        all_meshes.append(
            box([0.02, 1.10, 0.085], [-0.575, 0, z_slat], COLOR_WOOD)
        )
        all_meshes.append(
            box([0.02, 1.10, 0.085], [0.575, 0, z_slat], COLOR_WOOD)
        )
        all_meshes.append(
            box([1.10, 0.02, 0.085], [0, 0.575, z_slat], COLOR_WOOD)
        )

    # 5. Cassette Layers
    z_layer1 = z_sub_mesh + 0.025
    all_meshes.append(
        box([1.08, Y_L, 0.045], [0, Y_C, z_layer1], COLOR_ZEOLITE)
    )

    z_mesh1 = z_layer1 + 0.0255
    all_meshes.append(
        perforated_plate([1.12, Y_L + 0.02, 0.006], [0, Y_C - 0.01, z_mesh1])
    )

    z_layer2 = z_mesh1 + 0.025
    all_meshes.append(
        box([1.08, Y_L, 0.045], [0, Y_C, z_layer2], COLOR_BIOCHAR)
    )

    z_mesh2 = z_layer2 + 0.0255
    all_meshes.append(
        perforated_plate([1.12, Y_L + 0.02, 0.006], [0, Y_C - 0.01, z_mesh2])
    )

    z_layer3 = z_mesh2 + 0.018
    all_meshes.append(
        box([1.08, Y_L, 0.030], [0, Y_C, z_layer3], COLOR_VOC_PAD)
    )

    z_mesh_top = z_layer3 + 0.018
    all_meshes.append(
        perforated_plate([1.12, Y_L + 0.02, 0.006], [0, Y_C - 0.01, z_mesh_top])
    )

    # 6. Crop Tiers
    shelf_heights = [
        z_mesh_top + 0.05,
        z_mesh_top + 0.23,
        z_mesh_top + 0.41,
        z_mesh_top + 0.59,
    ]
    for z_sh in shelf_heights:
        all_meshes.append(
            perforated_plate(
                [1.10, 1.10, 0.008],
                [0, 0, z_sh],
                hole_r=0.015,
                grid_size=(10, 10),
            )
        )
        for xi in range(6):
            for yi in range(6):
                xp, yp = -0.42 + (xi * 0.168), -0.42 + (yi * 0.168)
                all_meshes.append(onion(0.040, [xp, yp, z_sh + 0.038]))

    # 7. Chimney Header
    z_header = z_mesh_top + 0.78
    all_meshes.append(
        box([1.22, 1.22, 0.012], [0, 0, z_header], COLOR_HEADER_METAL)
    )
    all_meshes.append(
        cyl(0.12, 0.20, [0, 0, z_header + 0.106], COLOR_HEADER_METAL)
    )
    all_meshes.append(
        cyl(0.14, 0.03, [0, 0, z_header + 0.206], COLOR_HEADER_METAL)
    )

    for angle in [0, 90, 180, 270]:
        rad = np.radians(angle)
        all_meshes.append(
            cyl(
                0.005,
                0.08,
                [0.11 * np.cos(rad), 0.11 * np.sin(rad), z_header + 0.246],
                COLOR_HEADER_METAL,
            )
        )

    all_meshes.append(
        cone(0.22, 0.08, [0, 0, z_header + 0.306], COLOR_HEADER_METAL)
    )

    for bx, by in [(-0.58, -0.58), (-0.58, 0.58), (0.58, -0.58), (0.58, 0.58)]:
        all_meshes.append(
            box([0.06, 0.06, 0.05], [bx, by, z_header], COLOR_HEADER_METAL)
        )
        kx = bx + (0.065 if bx > 0 else -0.065)
        all_meshes.append(
            cyl(0.020, 0.02, [kx, by, z_header], COLOR_PLASTIC_DARK)
        )

    joined_mesh = trimesh.util.concatenate(all_meshes)
    joined_mesh.export("onion_system_single_mesh.stl")

    scene = trimesh.Scene(joined_mesh)
    scene.export("onion_system_figure_twin.glb")
    print("3D Mesh files (.stl & .glb) exported successfully!")


# ==========================================
# PART 2: REAL-TIME AIRFLOW SIMULATION (PYGAME)
# ==========================================

WIDTH, HEIGHT = 700, 800
FPS = 60
NUM_PARTICLES = 50


def bezier_point(p0, p1, p2, p3, t):
    u = 1 - t
    tt = t * t
    uu = u * u
    x = uu * u * p0[0] + 3 * uu * t * p1[0] + 3 * u * tt * p2[0] + tt * t * p3[0]
    y = uu * u * p0[1] + 3 * uu * t * p1[1] + 3 * u * tt * p2[1] + tt * t * p3[1]
    return x, y


PATHS = [
    [
        (WIDTH * 0.38, HEIGHT * 0.94),
        (WIDTH * 0.22, HEIGHT * 0.65),
        (WIDTH * 0.47, HEIGHT * 0.20),
        (WIDTH * 0.20, HEIGHT * 0.04),
    ],
    [
        (WIDTH * 0.44, HEIGHT * 0.94),
        (WIDTH * 0.35, HEIGHT * 0.55),
        (WIDTH * 0.49, HEIGHT * 0.20),
        (WIDTH * 0.36, HEIGHT * 0.02),
    ],
    [
        (WIDTH * 0.50, HEIGHT * 0.94),
        (WIDTH * 0.50, HEIGHT * 0.55),
        (WIDTH * 0.50, HEIGHT * 0.20),
        (WIDTH * 0.50, HEIGHT * 0.01),
    ],
    [
        (WIDTH * 0.56, HEIGHT * 0.94),
        (WIDTH * 0.65, HEIGHT * 0.55),
        (WIDTH * 0.51, HEIGHT * 0.20),
        (WIDTH * 0.64, HEIGHT * 0.02),
    ],
    [
        (WIDTH * 0.62, HEIGHT * 0.94),
        (WIDTH * 0.78, HEIGHT * 0.65),
        (WIDTH * 0.53, HEIGHT * 0.20),
        (WIDTH * 0.80, HEIGHT * 0.04),
    ],
]


class AirParticle:

    def __init__(self, path):
        self.path = path
        self.t = random.uniform(0.0, 1.0)
        self.speed = random.uniform(0.003, 0.006)
        self.size = 14

    def update(self):
        self.t += self.speed
        if self.t > 1.0:
            self.t = 0.0

    def get_color(self):
        if self.t < 0.20:
            return (0, 210, 255)
        elif self.t < 0.70:
            return (255, 215, 0)
        else:
            return (255, 60, 0)

    def draw(self, surface):
        x, y = bezier_point(*self.path, self.t)
        nx, ny = bezier_point(*self.path, min(1.0, self.t + 0.01))

        angle = math.atan2(ny - y, nx - x)
        color = self.get_color()

        p1 = (
            x + self.size * math.cos(angle),
            y + self.size * math.sin(angle),
        )
        p2 = (
            x + self.size * 0.65 * math.cos(angle + 2.5),
            y + self.size * 0.65 * math.sin(angle + 2.5),
        )
        p3 = (
            x + self.size * 0.65 * math.cos(angle - 2.5),
            y + self.size * 0.65 * math.sin(angle - 2.5),
        )

        pygame.draw.polygon(surface, color, [p1, p2, p3])
        pygame.draw.polygon(surface, (0, 0, 0), [p1, p2, p3], 2)


def draw_prototype_structure(surface):
    surface.fill((245, 245, 247))

    pygame.draw.rect(
        surface, (35, 35, 35), (130, 680, 440, 45), border_radius=4
    )
    for x in range(150, 550, 30):
        pygame.draw.rect(surface, (160, 165, 170), (x, 690, 18, 25))

    pygame.draw.rect(surface, (235, 235, 225), (140, 650, 420, 15))
    pygame.draw.rect(surface, (15, 15, 15), (140, 630, 420, 15))
    pygame.draw.rect(surface, (200, 150, 95), (140, 610, 420, 15))

    pygame.draw.rect(surface, (185, 125, 68), (110, 180, 22, 490))
    pygame.draw.rect(surface, (185, 125, 68), (568, 180, 22, 490))

    for sy in [560, 460, 360, 260]:
        pygame.draw.rect(surface, (195, 200, 205), (132, sy, 436, 6))
        for ox in range(160, 540, 32):
            pygame.draw.circle(surface, (225, 140, 45), (ox, sy - 10), 9)
            pygame.draw.line(
                surface, (170, 100, 25), (ox, sy - 19), (ox, sy - 14), 2
            )

    pygame.draw.rect(surface, (45, 48, 52), (120, 170, 460, 12))
    pygame.draw.rect(surface, (45, 48, 52), (300, 100, 100, 70))
    pygame.draw.polygon(
        surface, (45, 48, 52), [(260, 100), (350, 55), (440, 100)]
    )


def run_simulation():
    pygame.init()
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("3D Prototype Airflow Simulation")
    clock = pygame.time.Clock()

    particles = [
        AirParticle(random.choice(PATHS)) for _ in range(NUM_PARTICLES)
    ]

    running = True
    while running:
        clock.tick(FPS)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        draw_prototype_structure(screen)

        for p in particles:
            p.update()
            p.draw(screen)

        pygame.display.flip()

    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    try:
        generate_3d_geometry()
    except Exception as e:
        print(f"3D Geometry generation skipped due to: {e}")
        
    run_simulation()

C:\Users\Chaitya\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Building 3D Mesh Geometry...
3D Mesh files (.stl & .glb) exported successfully!
